# Walmart Sales Data — End-to-End Python & SQL Project

This notebook takes a raw Walmart sales dataset from Kaggle, cleans it using pandas through a 6-stage cleaning process, and loads the cleaned data into a PostgreSQL database for further SQL analysis.

**Note on placeholders:** all usernames, file paths, and credentials in this notebook use generic placeholders (e.g. `YOUR_USERNAME`, `YOUR_PASSWORD`). Replace them with your own values before running, and never commit real credentials to a public repository.

## Step 1: Install required libraries
`sqlalchemy` and `psycopg2-binary` let Python talk to PostgreSQL. `kaggle` lets us download datasets directly via the Kaggle API.

In [ ]:
%pip install sqlalchemy pymysql psycopg2-binary kaggle

## Step 2: Set up Kaggle API access
The Kaggle API needs to know where your `kaggle.json` credentials file lives. This file contains your personal Kaggle API key — **never share or commit this file to GitHub.**

Replace `YOUR_USERNAME` below with your actual Windows username, or better, set this path using an environment variable so it's not hardcoded at all.

In [ ]:
import os

# Tell the Kaggle API where to find your credentials file (kaggle.json).
# Replace YOUR_USERNAME with your actual Windows username.
os.environ['KAGGLE_CONFIG_DIR'] = r'C:\Users\YOUR_USERNAME\.kaggle'

## Step 3: Move your Kaggle credentials into place
This copies your downloaded `kaggle.json` (from your Kaggle account settings) into the `.kaggle` folder the API expects. This only needs to be done once per machine.

In [ ]:
import os
import shutil

# Create the .kaggle folder if it doesn't already exist
os.makedirs(r'C:\Users\YOUR_USERNAME\.kaggle', exist_ok=True)

# Copy kaggle.json from Downloads into the .kaggle folder
# (Update the source path if your kaggle.json is saved somewhere else)
shutil.copy(
    r'C:\Users\YOUR_USERNAME\Downloads\kaggle.json',
    r'C:\Users\YOUR_USERNAME\.kaggle\kaggle.json'
)

## Step 4: Download the dataset from Kaggle
This uses the Kaggle API/CLI to pull the dataset directly, instead of manually downloading it from the website.

In [ ]:
!kaggle datasets download -d najir0123/walmart-10k-sales-datasets
!unzip walmart-10k-sales-datasets.zip

## Step 5: Extract the dataset (Windows-friendly alternative)
`!unzip` is a Linux/Mac command and doesn't work directly in a Windows terminal, so we use Python's built-in `zipfile` library instead to extract the downloaded zip file.

In [ ]:
import zipfile

# Replace YOUR_USERNAME with your actual Windows username
with zipfile.ZipFile(
    r'C:\Users\YOUR_USERNAME\Downloads\walmart-10k-sales-datasets.zip', 'r'
) as z:
    z.extractall(r'C:\Users\YOUR_USERNAME\Downloads')

## Step 6: Load the raw CSV into pandas
This is our starting point — the raw, unclean data, exactly as it came from Kaggle.

In [ ]:
import pandas as pd

# Replace YOUR_USERNAME with your actual Windows username
df = pd.read_csv(r'C:\Users\YOUR_USERNAME\Downloads\Walmart.csv')
print(df)

In [ ]:
# Preview the first few rows
df.head()

---
## Data Cleaning: Stage 1 — Understand the shape of the data
Before touching anything, we check how many rows/columns exist and what data type each column currently is.

In [ ]:
# Check number of rows and columns
df.shape

In [ ]:
# Check column names, non-null counts, and data types all at once
df.info()

---
## Data Cleaning: Stage 2 — Find and handle missing values
We check which columns have missing (null) values, and how many.

In [ ]:
# Count missing values per column
df.isnull().sum()

In [ ]:
# unit_price and quantity both showed 31 missing values.
# Before deciding to drop or fill them, we check whether the missing
# values are in the SAME rows, or scattered across different rows.
df[df['unit_price'].isnull() | df['quantity'].isnull()]

In [ ]:
# Check the total row count before making any changes,
# so we can measure the impact of dropping rows later.
len(df)

In [ ]:
# Decision: drop these rows.
# The missing values overlap in the same ~31 rows, which is only
# about 0.3% of the dataset. Since unit_price and quantity are core
# fields needed to calculate revenue, filling them with guessed
# values (like an average) risks introducing inaccurate revenue
# figures. Dropping is the safer, more defensible choice.
df = df.dropna(subset=['unit_price', 'quantity'], how='all')

In [ ]:
# Confirm the row count dropped by exactly the expected amount
len(df)

In [ ]:
# Double-check: no missing values remain anywhere in the dataset
df.isnull().sum()

---
## Data Cleaning: Stage 3 — Find and remove duplicates
We check for duplicate transactions — either full-row duplicates, or repeated invoice_id values (which should be unique per transaction).

In [ ]:
# Check for full-row duplicates
df.duplicated().sum()

In [ ]:
# Check for duplicate invoice_id values specifically
df['invoice_id'].duplicated().sum()

In [ ]:
# Both counts matched, meaning whenever an invoice_id repeats, the
# entire row repeats too -- these are genuine duplicate entries,
# not coincidental ID overlaps. Before dropping, confirm how many
# times each invoice_id appears (expecting a clean pattern of 2x,
# not 3x or more, which would suggest a bigger data problem).
df['invoice_id'].value_counts().head(10)

In [ ]:
len(df)

In [ ]:
# Drop duplicates, keeping only the first occurrence of each invoice_id
df = df.drop_duplicates(subset=['invoice_id'], keep='first')

In [ ]:
# Confirm no duplicates remain
df.duplicated().sum()

In [ ]:
# Confirm the new row count reflects the removed duplicates
len(df)

---
## Data Cleaning: Stage 4 — Fix data types
Some columns were read in as text (object) when they should be numeric, dates, or times. We fix each one so the data is actually usable for calculations and sorting.

In [ ]:
# Check current data types for every column
df.dtypes

### Fixing `unit_price`
This column shows as `object` (text) instead of a number — likely because of a `$` symbol in the values.

In [ ]:
# Confirm the $ symbol is present
df['unit_price'].head(10)

In [ ]:
# Remove the $ symbol and convert the column to a proper float type
df['unit_price'] = df['unit_price'].str.replace('$', '', regex=False).astype(float)

In [ ]:
# Verify the fix: should now show float64, with no $ signs
df['unit_price'].dtype
df['unit_price'].head(10)

### Fixing `date`
This column is stored as text. We first inspect its format before converting, since guessing the wrong day/month order can silently corrupt the dates.

In [ ]:
df['date'].head(10)

In [ ]:
# Check a wider sample of unique date values to confirm the exact
# format is consistent (DD/MM/YY) rather than mixed formats.
df['date'].unique()[:20]

In [ ]:
# Convert to a real datetime type, explicitly specifying the format
# (day/month/2-digit year) so pandas doesn't misread day vs month.
df['date'] = pd.to_datetime(df['date'], format='%d/%m/%y')

In [ ]:
# Verify the fix: should now show datetime64[ns]
df['date'].dtype
df['date'].head(10)

### Fixing `time`
This column is stored as text in HH:MM:SS format.

In [ ]:
df['time'].head(10)

In [ ]:
# Convert to a real time type. We first parse it as a full datetime
# (pandas requires this step), then extract just the time portion,
# since there's no date attached to these values.
df['time'] = pd.to_datetime(df['time'], format='%H:%M:%S').dt.time

In [ ]:
# Verify the fix: values are now real time objects, even though the
# dtype still displays as 'object' -- pandas has no dedicated
# time-only dtype, so this is expected and correct.
df['time'].dtype
df['time'].head(10)

In [ ]:
# Final check: confirm all data types are now correct across the board
df.dtypes

---
## Data Cleaning: Stage 5 — Standardize categorical values
We check text/categorical columns for inconsistent casing, spacing, or typos (e.g. 'walmart' vs 'Walmart' vs 'WALMART ') that would otherwise be treated as different categories.

In [ ]:
# Check Branch codes for consistency
df['Branch'].unique()

In [ ]:
# Check City names for consistency
df['City'].unique()

In [ ]:
# Check category values for consistency
df['category'].unique()

In [ ]:
# Check payment_method values for consistency
df['payment_method'].unique()

**Result:** all four categorical columns were already clean and consistent — no standardization needed.

---
## Data Cleaning: Stage 6 — Check outliers and logic errors
We check whether the numeric values make real-world sense (no negative prices, no impossible quantities, no broken dates, etc.).

In [ ]:
df.describe()

**Result:** all numeric ranges checked out as realistic. No negative prices, no zero/negative quantities, no dates outside the expected 2019–2023 range, and no runaway profit margins. The `rating` column's minimum of 3 (rather than 1) was noted as an observation, not an error.

---
## Pushing the cleaned data to PostgreSQL
Finally, we load the fully cleaned dataframe into a PostgreSQL database table, ready for SQL analysis.

**Security note:** never hardcode your real database password directly in a notebook you plan to share or upload. Below, the password is read from an environment variable instead, so it's never exposed in the notebook file itself.

Before running this cell, set the environment variable in PowerShell:
```powershell
$env:POSTGRES_PASSWORD = "your_actual_password"
```

In [ ]:
import os
from sqlalchemy import create_engine

# Credentials are read from environment variables instead of being
# hardcoded, so this notebook is safe to share/upload publicly.
username = os.environ.get('POSTGRES_USER', 'postgres')
password = os.environ.get('POSTGRES_PASSWORD')  # set this before running
host = os.environ.get('POSTGRES_HOST', 'localhost')
port = os.environ.get('POSTGRES_PORT', '5432')
database = os.environ.get('POSTGRES_DB', 'walmart_db')

if not password:
    raise ValueError(
        'POSTGRES_PASSWORD environment variable is not set. '
        'Set it before running this cell, e.g. in PowerShell:\n'
        '  $env:POSTGRES_PASSWORD = "your_actual_password"'
    )

connection_string = f'postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}'
engine = create_engine(connection_string)

# Load the cleaned dataframe into PostgreSQL, replacing any existing
# table with the same name (safe here since this is our full, final,
# cleaned dataset each time this cell runs).
df.to_sql('walmart', con=engine, if_exists='replace', index=False)
print('Data loaded successfully')